# 01 — Exploratory Data Analysis

**Goal:** Understand the dataset structure, class imbalance, and key signals before any modelling.

**Key questions:**
1. How severe is the class imbalance?
2. Which features show the strongest marginal association with fraud?
3. Are there data quality issues (nulls, outliers, cardinality problems)?

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from data_loader import load_raw, TARGET

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('muted')

## 1. Load & inspect

In [ ]:
df = load_raw()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nMissing values (%):')
print(df.isna().mean().mul(100).round(2).sort_values(ascending=False))

## 2. Class imbalance

In [ ]:
counts = df[TARGET].value_counts()
rates  = df[TARGET].value_counts(normalize=True).mul(100).round(2)

print('Class counts:')
print(counts.rename({0: 'Legitimate', 1: 'Fraud'}))
print('\nClass rates (%):')
print(rates.rename({0: 'Legitimate', 1: 'Fraud'}))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'tomato'], width=0.5)
ax.set_ylabel('Count')
ax.set_title('Class distribution')
for i, v in enumerate(counts.values):
    ax.text(i, v + 20, f'{rates.values[i]:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/01_class_distribution.png', bbox_inches='tight')
plt.show()

**Insight:** The dataset is heavily imbalanced (~1% fraud). This rules out accuracy as a useful metric and motivates PR-AUC, cost-sensitive thresholding, and `class_weight='balanced'` throughout.

## 3. Numeric feature distributions

In [ ]:
numeric_cols = ['amount', 'transaction_hour', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        ax.hist(df[df[TARGET] == label][col], bins=40, alpha=0.6,
                color=color, label='Legit' if label == 0 else 'Fraud', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle('Numeric feature distributions by class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/01_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Binary feature fraud rates

In [ ]:
binary_cols = ['foreign_transaction', 'location_mismatch']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col in zip(axes, binary_cols):
    rates = df.groupby(col)[TARGET].mean().mul(100)
    ax.bar([f'{col}=0', f'{col}=1'], rates.values, color=['steelblue', 'tomato'], width=0.5)
    ax.set_ylabel('Fraud rate (%)')
    ax.set_title(col.replace('_', ' ').title())
    for j, v in enumerate(rates.values):
        ax.text(j, v + 0.05, f'{v:.2f}%', ha='center', fontsize=10)

plt.suptitle('Fraud rate by binary features', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/01_binary_fraud_rates.png', bbox_inches='tight')
plt.show()

**Insight:** Location mismatch is the strongest single binary predictor — fraud rate ~8.4% vs ~0.86% without mismatch (~10× lift). Foreign transactions also show elevated risk. Neither feature alone is sufficient for classification, but both should be high-weight in any model.

## 5. Velocity quartile analysis

In [ ]:
velocity_analysis = (
    df.groupby(pd.qcut(df['velocity_last_24h'], q=4, labels=['Q1','Q2','Q3','Q4']))[TARGET]
    .mean().mul(100).round(3)
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(velocity_analysis.index, velocity_analysis.values, color='steelblue', width=0.6)
ax.set_xlabel('Velocity quartile (transactions in last 24h)')
ax.set_ylabel('Fraud rate (%)')
ax.set_title('Fraud rate by transaction velocity quartile')
plt.tight_layout()
plt.savefig('../reports/figures/01_velocity_fraud_rate.png', bbox_inches='tight')
plt.show()

print(velocity_analysis)

**Insight:** Fraud rate increases monotonically with transaction velocity, confirming that velocity captures genuine behavioural risk. This motivates the `amount_x_velocity` interaction feature in notebook 02.

## 6. Correlation heatmap

In [ ]:
corr_cols = numeric_cols + binary_cols + [TARGET]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation matrix (lower triangle)')
plt.tight_layout()
plt.savefig('../reports/figures/01_correlation_heatmap.png', bbox_inches='tight')
plt.show()

**Insight:** Features are largely uncorrelated with each other (no multicollinearity concerns). Correlations with `is_fraud` are all modest individually, which is expected — fraud detection is a multi-signal problem where combinations of weak predictors matter more than any single feature.

## 7. Merchant category fraud rates

In [ ]:
cat_rates = (
    df.groupby('merchant_category')[TARGET]
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'count': 'n_transactions'})
    .sort_values('fraud_rate', ascending=True)
    .assign(fraud_rate=lambda x: x['fraud_rate'].mul(100))
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cat_rates.index, cat_rates['fraud_rate'], color='steelblue')
ax.set_xlabel('Fraud rate (%)')
ax.set_title('Fraud rate by merchant category')
plt.tight_layout()
plt.savefig('../reports/figures/01_merchant_category_fraud.png', bbox_inches='tight')
plt.show()

print(cat_rates)

## EDA Summary

| Finding | Implication |
|---|---|
| ~1% fraud rate | Use PR-AUC, not accuracy; apply class weighting |
| Location mismatch: 10× lift | Strong binary predictor |
| Velocity monotonically predictive | Engineer interaction features |
| No severe multicollinearity | All features safe to include |
| Merchant category varies | One-hot encoding will add signal |